### SET UP

In [ ]:
import os
import sys
import gc
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
import pandas as pd
import numpy as np
from sklearn.metrics import matthews_corrcoef, roc_auc_score

sys.path.append(os.path.abspath("../"))
from core.module05_fusion_classifier.dataset import VariantFusionDataset
from core.module05_fusion_classifier.fusion_model import MultiStrategyFusionModel
from core.module05_fusion_classifier.xgboost_model import XGBoostFusionManager

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[*] Đang sử dụng thiết bị: {DEVICE}")

# ==============================================================================
# 1. CẤU HÌNH VÀ ĐƯỜNG DẪN
# ==============================================================================
BASE_DIR = r"D:/variant_data"
BIO_DIR = f"{BASE_DIR}/processed_parquet"
GEOM_DIR = f"{BASE_DIR}/geometry"
EMBED_DIR = f"{BASE_DIR}/embeddings"
MODELS_DIR = f"{BASE_DIR}/saved_models"

os.makedirs(MODELS_DIR, exist_ok=True)

CONFIG = {
    "dna_model": "evo2_1b_center",
    "dna_dim": 2048,
    "prot_model": "esm2_650m_center",
    "prot_dim": 1280,
    "batch_size": 256,
    "epochs": 15,
    "lr": 1e-4,
    "num_workers": 0 # [BẢN VÁ] Set = 0 để tránh lỗi Multiprocessing trên Windows/Jupyter
}

EXPERIMENTS = [
    {"name": "PyTorch_Concat", "type": "pytorch", "fusion": "concat"},
    {"name": "PyTorch_Gating", "type": "pytorch", "fusion": "gating"},
    {"name": "Pure_XGBoost_Concat", "type": "xgboost_pure", "fusion": "concat"},
    {"name": "Hybrid_CrossAttn_XGBoost", "type": "hybrid", "fusion": "cross_attention"}
]

# ==============================================================================
# 2. CÁC HÀM TIỆN ÍCH (HELPERS)
# ==============================================================================
def get_dataloaders(split_name, is_train=True):
    dataset = VariantFusionDataset(
        bio_parquet_path=f"{BIO_DIR}/{split_name}_normalized.parquet",
        dna_geom_path=f"{GEOM_DIR}/{split_name}/{CONFIG['dna_model']}_geom_norm.parquet",
        prot_geom_path=f"{GEOM_DIR}/{split_name}/{CONFIG['prot_model']}_geom_norm.parquet",
        dna_pt_path=f"{EMBED_DIR}/{split_name}/{CONFIG['dna_model']}.pt",
        prot_pt_path=f"{EMBED_DIR}/{split_name}/{CONFIG['prot_model']}.pt",
        is_train=True # Bật True để luôn load Label (ngay cả tập Test cũng cần label để tính MCC)
    )
    return DataLoader(dataset, batch_size=CONFIG["batch_size"], shuffle=is_train, 
                      num_workers=CONFIG["num_workers"], pin_memory=True)

def evaluate_pytorch(model, dataloader, criterion):
    """Đánh giá mô hình PyTorch, trả về Loss và MCC."""
    model.eval()
    total_loss = 0
    all_preds, all_labels = [], []
    
    with torch.no_grad():
        for batch in dataloader:
            v_dna = batch["v_dna"].to(DEVICE)
            v_prot = batch["v_prot"].to(DEVICE)
            bio = batch["bio_features"].to(DEVICE)
            geom = batch["geom_features"].to(DEVICE)
            labels = batch["label"].to(DEVICE)
            
            logits = model(v_dna, v_prot, bio, geom, return_features=False)
            loss = criterion(logits, labels)
            total_loss += loss.item()
            
            # Tính xác suất và nhãn dự đoán
            probs = torch.sigmoid(logits)
            preds = (probs > 0.5).float()
            
            all_preds.append(preds.cpu())
            all_labels.append(labels.cpu())
            
    avg_loss = total_loss / len(dataloader)
    
    # Ép về 1D để tính MCC
    preds_np = torch.cat(all_preds).squeeze(-1).numpy()
    labels_np = torch.cat(all_labels).squeeze(-1).numpy()
    mcc = matthews_corrcoef(labels_np, preds_np)
    
    return avg_loss, mcc

def extract_features(model, dataloader, extract_f_global=True):
    """Trích xuất data ra Numpy cho XGBoost."""
    model.eval()
    all_f_global, all_v_dna, all_v_prot, all_bg, all_labels = [], [], [], [], []
    
    with torch.no_grad():
        for batch in dataloader:
            v_dna = batch["v_dna"].to(DEVICE)
            v_prot = batch["v_prot"].to(DEVICE)
            bg = torch.cat([batch["bio_features"], batch["geom_features"]], dim=-1).to(DEVICE)
            
            # [BẢN VÁ 3] Bỏ qua qua mạng Neural Network nếu không cần thiết
            if extract_f_global:
                f_global = model(v_dna, v_prot, batch["bio_features"].to(DEVICE), 
                                 batch["geom_features"].to(DEVICE), return_features=True)
                all_f_global.append(f_global.cpu())
                
            all_v_dna.append(v_dna.cpu())
            all_v_prot.append(v_prot.cpu())
            all_bg.append(bg.cpu())
            
            if "label" in batch:
                # [BẢN VÁ 2] Squeeze label từ (Batch, 1) về (Batch,)
                all_labels.append(batch["label"].squeeze(-1).cpu())
                
    f_glob_out = torch.cat(all_f_global).numpy() if extract_f_global else None
    return (f_glob_out, torch.cat(all_v_dna).numpy(), torch.cat(all_v_prot).numpy(), 
            torch.cat(all_bg).numpy(), torch.cat(all_labels).numpy() if all_labels else None)

### The Master Loop

In [ ]:
print("[*] Đang nạp DataLoaders lên RAM...")
train_loader = get_dataloaders("train", is_train=True)
val_loader = get_dataloaders("val", is_train=False)
test_loader = get_dataloaders("test_clinvar_hq", is_train=False)

results_log = [] # Lưu kết quả Test của tất cả cấu hình

for exp in EXPERIMENTS:
    exp_name = exp["name"]
    print("\n" + "="*80)
    print(f"[>>>] BẮT ĐẦU THÍ NGHIỆM: {exp_name.upper()} [<<<]")
    print("="*80)
    
    # Khởi tạo giá trị rỗng cho Garbage Collector an toàn
    xgb_manager = None; best_model_path = f"{MODELS_DIR}/{exp_name}_best.pth"
    dna_tr = None; prot_tr = None; bg_tr = None; y_tr = None; f_glob_tr = None
    dna_vl = None; prot_vl = None; bg_vl = None; y_vl = None; f_glob_vl = None
    dna_ts = None; prot_ts = None; bg_ts = None; y_ts = None; f_glob_ts = None
    
    model = MultiStrategyFusionModel(
        dna_in_dim=CONFIG["dna_dim"], prot_in_dim=CONFIG["prot_dim"], fusion_strategy=exp["fusion"]
    ).to(DEVICE)
    
    test_mcc = 0.0
    
    # -------------------------------------------------------------------------
    # LUỒNG 1: PYTORCH END-TO-END
    # -------------------------------------------------------------------------
    if exp["type"] == "pytorch":
        optimizer = torch.optim.AdamW(model.parameters(), lr=CONFIG["lr"])
        criterion = nn.BCEWithLogitsLoss()
        best_val_mcc = -1.0
        
        for epoch in range(CONFIG["epochs"]):
            model.train()
            total_train_loss = 0
            for batch in train_loader:
                v_dna, v_prot = batch["v_dna"].to(DEVICE), batch["v_prot"].to(DEVICE)
                bio, geom = batch["bio_features"].to(DEVICE), batch["geom_features"].to(DEVICE)
                labels = batch["label"].to(DEVICE)
                
                optimizer.zero_grad()
                logits = model(v_dna, v_prot, bio, geom, return_features=False)
                loss = criterion(logits, labels)
                loss.backward()
                optimizer.step()
                total_train_loss += loss.item()
                
            val_loss, val_mcc = evaluate_pytorch(model, val_loader, criterion)
            print(f"  -> Epoch [{epoch+1}/{CONFIG['epochs']}] Train Loss: {total_train_loss/len(train_loader):.4f} | Val Loss: {val_loss:.4f} | Val MCC: {val_mcc:.4f}")
            
            # [BẢN VÁ 4] Lưu Best Model (Early Stopping cho PyTorch)
            if val_mcc > best_val_mcc:
                best_val_mcc = val_mcc
                torch.save(model.state_dict(), best_model_path)
                print(f"     [+] Đã lưu mô hình tốt nhất (MCC: {best_val_mcc:.4f})")
                
        # Load lại mô hình tốt nhất để Test
        model.load_state_dict(torch.load(best_model_path))
        _, test_mcc = evaluate_pytorch(model, test_loader, criterion)
        print(f"\n  => TEST MCC KẾT QUẢ: {test_mcc:.4f}")
        
    # -------------------------------------------------------------------------
    # LUỒNG 2: XGBOOST THUẦN (PURE ML)
    # -------------------------------------------------------------------------
    elif exp["type"] == "xgboost_pure":
        _, dna_tr, prot_tr, bg_tr, y_tr = extract_features(model, train_loader, extract_f_global=False)
        _, dna_vl, prot_vl, bg_vl, y_vl = extract_features(model, val_loader, extract_f_global=False)
        _, dna_ts, prot_ts, bg_ts, y_ts = extract_features(model, test_loader, extract_f_global=False)
        
        xgb_manager = XGBoostFusionManager(pca_components=256)
        xgb_manager.train_pure(dna_tr, prot_tr, bg_tr, y_tr, dna_vl, prot_vl, bg_vl, y_vl)
        
        # Test Inference
        _, _, metrics = xgb_manager.predict_pure(dna_ts, prot_ts, bg_ts, y_ts)
        test_mcc = metrics["MCC"]
        xgb_manager.save_model(MODELS_DIR, exp_name)
        print(f"\n  => TEST MCC KẾT QUẢ: {test_mcc:.4f}")
        
    # -------------------------------------------------------------------------
    # LUỒNG 3: HYBRID (PYTORCH TRÍCH XUẤT -> XGBOOST)
    # -------------------------------------------------------------------------
    elif exp["type"] == "hybrid":
        optimizer = torch.optim.AdamW(model.parameters(), lr=CONFIG["lr"])
        criterion = nn.BCEWithLogitsLoss()
        
        print("  [Hybrid Bước 1] Huấn luyện Proxy PyTorch (1 Epoch nhanh)...")
        # Với Hybrid, ta chỉ cần PyTorch học sơ qua để định hình khối Fusion (Train 1-2 epoch là đủ)
        for _ in range(2): 
            for batch in train_loader:
                v_dna, v_prot = batch["v_dna"].to(DEVICE), batch["v_prot"].to(DEVICE)
                labels = batch["label"].to(DEVICE)
                optimizer.zero_grad()
                loss = criterion(model(v_dna, v_prot, batch["bio_features"].to(DEVICE), batch["geom_features"].to(DEVICE)), labels)
                loss.backward()
                optimizer.step()
                
        print("  [Hybrid Bước 2] Trích xuất F_global...")
        f_glob_tr, _, _, _, y_tr = extract_features(model, train_loader, extract_f_global=True)
        f_glob_vl, _, _, _, y_vl = extract_features(model, val_loader, extract_f_global=True)
        f_glob_ts, _, _, _, y_ts = extract_features(model, test_loader, extract_f_global=True)
        
        print("  [Hybrid Bước 3] Huấn luyện XGBoost Classifier...")
        xgb_manager = XGBoostFusionManager()
        xgb_manager.train_hybrid(f_glob_tr, y_tr, f_glob_vl, y_vl)
        
        _, _, metrics = xgb_manager.predict_hybrid(f_glob_ts, y_ts)
        test_mcc = metrics["MCC"]
        xgb_manager.save_model(MODELS_DIR, exp_name)
        print(f"\n  => TEST MCC KẾT QUẢ: {test_mcc:.4f}")

    # Cập nhật bảng kết quả
    results_log.append({"Experiment": exp_name, "Test_MCC": test_mcc})

    # -------------------------------------------------------------------------
    # [BẢN VÁ 1] QUẢN LÝ BỘ NHỚ AN TOÀN TUYỆT ĐỐI
    # -------------------------------------------------------------------------
    del model, optimizer, criterion
    # Xóa tham chiếu, GC sẽ tự động dọn RAM mà không bao giờ báo lỗi NameError
    xgb_manager = dna_tr = prot_tr = bg_tr = y_tr = f_glob_tr = None
    dna_vl = prot_vl = bg_vl = y_vl = f_glob_vl = None
    dna_ts = prot_ts = bg_ts = y_ts = f_glob_ts = None
    
    torch.cuda.empty_cache()
    gc.collect()
    print(f"[*] Đã giải phóng RAM/VRAM.")

# ==============================================================================
# 4. TỔNG KẾT VÀ BÁO CÁO KẾT QUẢ
# ==============================================================================
print("\n" + "="*80)
print("[THÀNH CÔNG] BẢNG XẾP HẠNG THÍ NGHIỆM (LEADERBOARD)")
print("="*80)
df_results = pd.DataFrame(results_log).sort_values(by="Test_MCC", ascending=False)
print(df_results.to_markdown(index=False))
df_results.to_csv(f"{MODELS_DIR}/experiment_results.csv", index=False)